# Lab 2 — Advanced SQL Querying, Aggregations & AI Functions

**Goal:** Practice aggregations, window functions, and (where enabled) Databricks AI sentiment analysis.

**Workspace:** This notebook uses the `workspace` catalog and `lab_db` schema from Lab 1.

**Run order:** Execute cells from top to bottom. The AI function cell is optional because availability depends on workspace/compute permissions and product configuration.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.lab_db;

CREATE OR REPLACE TABLE workspace.lab_db.review_logs (
  review_id INT,
  customer_id INT,
  rating INT,
  review_text STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.lab_db.review_logs VALUES
  (1, 101, 5, 'Absolute best product! Fast delivery and great support.'),
  (2, 102, 1, 'Horrible experience. Package arrived damaged and support was rude.'),
  (3, 101, 4, 'Good product and quick delivery.'),
  (4, 103, 3, 'Average quality, but it works.'),
  (5, 104, 2, 'Late delivery and disappointing packaging.');

In [0]:
%sql
-- Inspect sample data
SELECT * FROM workspace.lab_db.review_logs ORDER BY review_id;

In [0]:
%sql
-- Window function: rank reviews by rating, highest first
SELECT
  review_id,
  customer_id,
  rating,
  DENSE_RANK() OVER (ORDER BY rating DESC) AS rating_rank,
  ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY rating DESC, review_id) AS customer_review_number
FROM workspace.lab_db.review_logs
ORDER BY rating_rank, review_id;

In [0]:
%sql
-- Aggregation exercise: average rating and review count per customer
SELECT
  customer_id,
  COUNT(*) AS review_count,
  ROUND(AVG(rating), 2) AS average_rating
FROM workspace.lab_db.review_logs
GROUP BY customer_id
ORDER BY average_rating DESC;

## Optional: AI sentiment function

Run this only if your workspace supports `ai_analyze_sentiment` and your identity has permission. If it fails with an unsupported-function or permission error, the earlier SQL exercises are still complete.

In [0]:
%sql
SELECT
  review_id,
  rating,
  ai_analyze_sentiment(review_text) AS sentiment_category
FROM workspace.lab_db.review_logs
ORDER BY review_id;

In [0]:
%sql
-- Verification: expected 5 review rows
SELECT COUNT(*) AS review_rows FROM workspace.lab_db.review_logs;